# 2026 Symbolic regression Monod paper
# Pretreating real data and Generating simulated data

In [ ]:
cwd = 

In [ ]:
from os import listdir

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

## Simple functions

In [ ]:
# smoothing

halfwindow = 3

def summingaround(N, halfwindow = 3):

    AA = N.copy()
    
    for w in np.arange(halfwindow) + 1: # summing over (2*halfwindow)-window
        BB = np.roll(AA, shift=-w, axis=0)
        BB[-w:] = 0
        AA = AA + BB
        BB = np.roll(AA, shift=w, axis=0)
        BB[:w] = 0
        AA = AA + BB
    
    return AA


def smoothing(N, halfwindow = 3):
    
    AA = summingaround(N, halfwindow=halfwindow)
    BB = np.ones(AA.shape)
    BB = summingaround(BB, halfwindow=halfwindow)
    
    return AA / BB

In [ ]:
# Compute per-capita derivatives

def to_rhos(T, N):
    
    if len(T.shape) == 1:
        assert T.shape[0] == N.shape[0], "T-array has length {} but N-array has length {}!".format(T.shape[0], N.shape[0])
        for axis in N.shape[1:]:
            T = T[...,np.newaxis]
    else:
        assert T.shape == N.shape, "T-array has shape {} but N-array has shape {}!".format(T.shape, N.shape)
    
    if T.dtype == int:
        T = T.astype(float)
    
    # (delta N) / N
    
    # forward roll for N
    A = np.roll(N, shift=-1, axis=0)
    A[-1:] = np.nan
    A = A - N
    A[np.isnan(A)] = 0

    # backward roll for N
    B = np.roll(N, shift=1, axis=0)
    B[:1] = np.nan
    B = B - N
    B[np.isnan(B)] = 0
    
    NNN = N.copy()
    NNN[N!=0] = (A[N!=0] - B[N!=0]) / N[N!=0]

    # delta T
    
    # forward roll for T
    A = np.roll(T, shift=-1, axis=0)
    A[-1:] = np.nan
    A = A - T
    A[np.isnan(A)] = 0

    # backward roll for T
    B = np.roll(T, shift=1, axis=0)
    B[:1] = np.nan
    B = B - T
    B[np.isnan(B)] = 0

    T = A - B

    return NNN / T

In [ ]:
# Compute AUCs/integrals
# returns an array of same dimensions,
# with first time point being zero

def to_AUCs(T, N, t0 = 0 # t0: if integer, reference index for integral computation, otherwise the raw AUC is returned
           ):
    
    if len(T.shape) == 1:
        assert T.shape[0] == N.shape[0], "T-array has length {} but N-array has length {}!".format(T.shape[0], N.shape[0])
        for axis in N.shape[1:]:
            T = T[...,np.newaxis]
    else:
        assert T.shape == N.shape, "T-array has shape {} but N-array has shape {}!".format(T.shape, N.shape)
    
    A = ( np.roll(N, shift=1, axis=0) + N ) / 2 # average N
#    A[0] = N[0] # not useful since replaced immediately

    B = T - np.roll(T, shift=1, axis=0) # delta T
    
    A *= B # average N * delta T
    A[0] = 0 # cut off first value, # Flooring is not required when using trapezoidal rule
    
    B = np.cumsum(A, axis=0) # cumulative sum
    
    return B



# Experimental data in R2A

## Check variation among biological replicates

In [ ]:
df = pd.DataFrame()
for file in sorted(listdir("{}/raws".format(cwd))):
    if (file[:3] == "Rs0") and (not "3031" in file ) and (not "3237" in file ):
        strn = file[file.index("i")+1:file.index(".csv")]
        data = pd.read_csv("{}/raws/{}".format(cwd, file), )
        data["strn"] = strn
        df = pd.concat([df, data])

M = df.set_index(["conc", "Time", "strn"]).values
print("median interreplicate-CV =", round(np.median(M.std(axis = 1) / M.mean(axis = 1)), 3), )
print("mean interreplicate-CV =", round(np.mean(M.std(axis = 1) / M.mean(axis = 1)), 3), )

## Pretreat real data

### Rs0 datasets: HAMBI species in R2A

In [ ]:
for name in listdir("{}/raws".format(cwd)):

    if name[:3] != "Rs0":
        continue

    df = pd.read_csv("{}/raws/{}".format(cwd, name), index_col=[0,1] )
    data = pd.DataFrame()
    concs = sorted(list(df.index.get_level_values("conc").unique()))
    
    for conc in concs:
        
        N = df.xs(conc, level="conc")
        N = N.sort_index()

        T = np.array(N.index, dtype=float)
        if T.max() > 100000:
            T = T / 60 / 60 # convert time unit from seconds to hours
        N = np.mean(N.to_numpy(), axis=1) # mean across replicates
        N = smoothing(N) # smoothing
        Nc = to_AUCs(T, N, t0=0)
        F = np.exp(-Nc)
        R = to_rhos(T, N)

        dataframe = pd.DataFrame({"T": T, "n": N, "Nc": Nc, "F": F, "U": np.exp(- T), "rho": R, "C": conc, }, )
        data = pd.concat([data, dataframe])
        
    data.to_csv("{}/data/{}".format(cwd, name), sep=",", index=False)

### RsB dataset: E. coli data in Glc + NH4+ (Held2024PNAS)

In [ ]:
df = pd.read_csv("{}/raws/Ecoli/manhart_colim_OD.csv".format(cwd), index_col=[0,2,3,4] )
df = df.groupby(["t", "C1", "C2",]).mean()  # mean across replicates

data = pd.DataFrame()
for C1 in sorted(list(df.index.get_level_values("C1").unique())):
    for C2 in sorted(list(df.xs(C1, level="C1").index.get_level_values("C2").unique())):
        
        N = df.xs((C1, C2,), level=["C1", "C2",])
        T = np.array(N.index, dtype=float) # convert time unit from seconds to hours
        N = smoothing(N.to_numpy()[:,0]) # smoothing
        Nc = to_AUCs(T, N, t0=0)
        F = np.exp(-Nc)
        R = to_rhos(T, N)

        dataframe = pd.DataFrame({"T": T, "n": N, "Nc": Nc, "F": F, "U": np.exp(- T), "rho": R, "C1": C1, "C2": C2, }, )
        data = pd.concat([data, dataframe])
        
data.to_csv("{}/data/RsBi0.csv".format(cwd,), sep=",", index=False)

## Visualisation of growth curves Fig.S1

In [ ]:
code = "Rs0"

strns = []
for name in listdir("{}/data".format(cwd)):
    if name[:3] == "Rs0":
        strns.append(name[4:-4])
strns.sort(key=int)

In [ ]:
nplots = 2 # nb of mini-plots per strain
nranks = 2 # nb of meta-columns

YYY = len(strns) // nranks
XXX = nplots * nranks

cmap = mpl.colormaps["viridis"]

In [ ]:
fig = plt.figure(figsize=(10,8), constrained_layout=True)
fig.supxlabel("time")
fig.supylabel("bacterial species (HAMBI code)")

gs = fig.add_gridspec(YYY, XXX) # (y, x)
axs = gs.subplots()

for i, strn in enumerate(strns): # go through strains
    strn = str(strn)
    rank = i // YYY
    row = i % YYY
    
    data = pd.read_csv("{}/data/{}i{}.csv".format(cwd, code, strn), sep=",")
    concs = sorted(list(data["C"].apply(float).unique()))

    for k, y in enumerate(["n", "rho"]):
        ax = axs[row,nplots*rank+k]
        
        for c, conc in enumerate(concs):
            colour = cmap(c / len(concs))
    
            datc = data[data["C"] == conc]
            xs = [datc["T"].to_numpy(), datc["T"].to_numpy(), np.arange(0, 5000, 1)]
            ax.plot(xs[k], datc[y], c=colour)
            
        ax.tick_params(axis="both", which="both", bottom=False, top=False, left=False, right=False, labelbottom=False, labeltop=False, labelleft=False, labelright=False, )
        ax.set(xlim = (0, np.max(xs[k])), #ylim = (0, data[y].max() * 1.1),
               ylabel = str(strn) if k == 0 else None,
               title = [r"population size $N$", r"per capita growth rate $\rho_{\rm obs}$", ][k] if row == 0 else None, )

### SAVING
name = "Time_dependence_{}.pdf".format(code)
plt.savefig("{}/plot/{}".format(cwd, name), facecolor='w', edgecolor='w', transparent=False, bbox_inches="tight")
plt.show()

# Simulations

In [ ]:
from scipy.integrate import solve_ivp

## Extract core values in experimental data

In [ ]:
from numpy.random import default_rng
rng = default_rng()

In [ ]:
df = pd.DataFrame()
for file in sorted(listdir("{}/data".format(cwd))):
    if file[:3] == "Rs0":
        strn = file[file.index("i")+1:file.index(".csv")]
        data = pd.read_csv("{}/data/{}".format(cwd, file), )
        if data["C"].max() > 100:
            data["C"] /= 100
        data["strn"] = strn
        df = pd.concat([df, data])

In [ ]:
df["C_round"] = df["C"].apply(lambda x: np.round(x, 3))
df["T_round"] = df["T"].apply(lambda x: np.round(x, 1))

In [ ]:
data_min = df.groupby(["T_round", "C_round", ])["n"].min().unstack()#.iloc[:,[0,-1]]
data_max = df.groupby(["T_round", "C_round", ])["n"].max().unstack()
data_mdn = df.groupby(["C_round", "T_round", ])["n"].median()
data_min_max = pd.concat({"Nmn": data_min, "Nmx": data_max, }, names = ["lvl"], axis = 1)
data_min_max = np.around(data_min_max.swaplevel(axis = 1).sort_index(axis = 1), 2)

In [ ]:
data_mdn.index.get_level_values("C_round").unique()

## Simulation models

In [ ]:
# resource-consumer model for simulation

def Monod_mult(t, X, nus, Ks, gmax, q, m, ):
    
    S, N = X[:-1], X[-1]
    dX = np.zeros(X.shape)
    dX[:-1] = - nus * S * N # dynamics of the substrates
    dX[-1] = q / (q + np.exp(- m * t)) * gmax * np.prod( S / (Ks + S) ) * N # dynamics of the population

    return dX

In [ ]:
# resource-consumer model for simulation

def Monod_addt(t, X, nus, Ks, gmax, q, m, ):
    
    S, N = X[:-1], X[-1]
    dX = np.zeros(X.shape)
    dX[:-1] = - nus * S * N # dynamics of the substrates
    dX[-1] = q / (q + np.exp(- m * t)) * gmax * np.sum( S / (Ks + S) ) * N # dynamics of the population

    return dX

In [ ]:
# resource-consumer model for simulation

def Monod_mult_unadjusted(t, X, nus, Ks, gmax, ):
    
    S, N = X[:-1], X[-1]
    dX = np.zeros(X.shape)
    dX[:-1] = - nus * S * N # dynamics of the substrates
    dX[-1] = gmax * np.prod( S / (Ks + S) ) * N # dynamics of the population

    return dX

In [ ]:
# resource-consumer model for simulation

def Monod_addt_unadjusted(t, X, nus, Ks, gmax, ):
    
    S, N = X[:-1], X[-1]
    dX = np.zeros(X.shape)
    dX[:-1] = - nus * S * N # dynamics of the substrates
    dX[-1] = gmax * np.sum( S / (Ks + S) ) * N # dynamics of the population

    return dX

In [ ]:
nus, Ks, gmax, q, m = 8.26047586e-01, 3.18891107e+00, 1.64494226e+00, 3.72080271e-04, 7.24529548e-01

In [ ]:
# Simulate growth curves at each concentration
def simulation(fun, # = Monod_mult,
               TTT,
               args = (nus, Ks, gmax, q, m,),
              ):
    
    Ns = np.zeros(len(TTT))
    ntimes = int(len(TTT) / len(concs))
    
    for c, conc in enumerate(concs):
        T = TTT[c*ntimes:(c+1)*ntimes]
        X0 = 1 if isinstance(args[0], float) else len(args[0])
        X0 = np.array([conc] * X0 + [0.1])
        sol = solve_ivp(fun, (T[0], T[-1]), X0, t_eval = T, args = args, )
        if sol.success:
            Ns[c*ntimes:(c+1)*ntimes] = sol.y[-1]
    
    return Ns

## Fitting and simulation of each model

In [ ]:
nrescs = 1

T = np.arange(48 * 6 + 1) / 6 # timepoints as in experiments
concs = (4 / 5) ** np.arange(20)[::-1] * 200 / 100 # concentrations as in experiments

### Simulation of 1D-Monod data

In [ ]:
from scipy.optimize import curve_fit

popt, _ = curve_fit(lambda X, nus, Ks, gmax, q, m: simulation(Monod_mult, X, args = (nus, Ks, gmax, q, m), ),
                    xdata = data_mdn.index.get_level_values("T_round").values, ydata = data_mdn, bounds=(0, np.inf), )

nus, Ks, gmax, q, m = popt
name = "Ms{}_nu={}_K={}_gmax={}_q={}_m={}".format(1, nus, Ks, gmax, q, m, )
print(name)
print(popt)

In [ ]:
nus, Ks, gmax, q, m = 8.26047586e-01, 3.18891107e+00, 1.64494226e+00, 3.72080271e-04, 7.24529548e-01 # fitted parameters

noise = 0.2

nus *= rng.lognormal(sigma = noise, size = 1)
Ks *= rng.lognormal(sigma = noise, size = 1)
gmax, q, m = np.array([gmax, q, m, ]) * rng.lognormal(sigma = noise, size = 3)
name = "Ms{}_nu={}_K={}_gmax={}_q={}_m={}".format(len(nus), "_".join([str(x) for x in nus]), "_".join([str(x) for x in Ks]), gmax, q, m, )
print(name)

for c, conc in enumerate(concs):
    
    X0 = 1 if isinstance(nus, float) else len(nus)
    X0 = np.array([conc] * X0 + [0.1])
    
    sol = solve_ivp(Monod_mult,
                    (T[0], T[-1]),
                    X0,
                    t_eval = T,
                    args = (nus, Ks, gmax, q, m,),
                    )
    N = sol.y[-1]
    Nc = to_AUCs(T, N, t0=0)
    F = np.exp(-Nc)
    R = to_rhos(T, N)

    dataframe = pd.DataFrame({"T": T, "n": N, "Nc": Nc, "F": F, "U": np.exp(- T), "rho": R, "C": conc, }, )
    data = dataframe if c == 0 else pd.concat([data, dataframe])
    
    plt.plot(T, N)
plt.show()

data.to_csv("{}/data_simulated_FR/{}.csv".format(cwd, name), sep=",", index=False)

### Simulation of multiplicative 2D-Monod data

In [ ]:
from scipy.optimize import curve_fit

popt, _ = curve_fit(lambda X, nus, Ks, gmax, q, m: simulation(Monod_mult, X, args = (np.array([nus, nus]), np.array([Ks, Ks]), gmax, q, m), ),
                    xdata = data_mdn.index.get_level_values("T_round").values, ydata = data_mdn, bounds=(0, np.inf), )

nus, Ks, gmax, q, m = popt
name = "Ms{}_nu={}_K={}_gmax={}_q={}_m={}".format(1, nus, Ks, gmax, q, m, )
print(name)
print(popt)

In [ ]:
nus, Ks, gmax, q, m = 0.70248826, 0.19589854, 0.31826243, 0.00106463, 0.67793093 # fitted parameters

noise = 0.2

nus = np.array([nus, nus]) * rng.lognormal(sigma = noise, size = 2)
Ks = np.array([Ks, Ks]) * rng.lognormal(sigma = noise, size = 2)
gmax, q, m = np.array([gmax, q, m, ]) * rng.lognormal(sigma = noise, size = 3)
name = "Ms{}_nu={}_K={}_gmax={}_q={}_m={}".format(len(nus), "_".join([str(x) for x in nus]), "_".join([str(x) for x in Ks]), gmax, q, m, )
print(name)

for c, conc in enumerate(concs):
    
    X0 = 1 if isinstance(nus, float) else len(nus)
    X0 = np.array([conc] * X0 + [0.1])
    
    sol = solve_ivp(Monod_mult,
                    (T[0], T[-1]),
                    X0,
                    t_eval = T,
                    args = (nus, Ks, gmax, q, m,),
                    )
    N = sol.y[-1]
    Nc = to_AUCs(T, N, t0=0)
    F = np.exp(-Nc)
    R = to_rhos(T, N)

    dataframe = pd.DataFrame({"T": T, "n": N, "Nc": Nc, "F": F, "U": np.exp(- T), "rho": R, "C": conc, }, )
    data = dataframe if c == 0 else pd.concat([data, dataframe])
    
    plt.plot(T, N)
plt.show()

data.to_csv("{}/data_simulated/{}.csv".format(cwd, name), sep=",", index=False)

### Simulation of additive 2D-Monod data

In [ ]:
from scipy.optimize import curve_fit

popt, _ = curve_fit(lambda X, nus, Ks, gmax, q, m: simulation(Monod_addt, X, args = (np.array([nus, nus]), np.array([Ks, Ks]), gmax, q, m), ),
                    xdata = data_mdn.index.get_level_values("T_round").values, ydata = data_mdn, bounds=(0, np.inf), )

nus, Ks, gmax, q, m = popt
name = "As{}_nu={}_K={}_gmax={}_q={}_m={}".format(1, nus, Ks, gmax, q, m, )
print(name)
print(popt)

In [ ]:
nus, Ks, gmax, q, m = 8.38874989e-01, 3.86999496e+00, 1.04200753e+00, 5.42190219e-04, 6.74326085e-01 # fitted parameters

noise = 0.2

nus = np.array([nus, nus]) * rng.lognormal(sigma = noise, size = 2)
Ks = np.array([Ks, Ks]) * rng.lognormal(sigma = noise, size = 2)
gmax, q, m = np.array([gmax, q, m, ]) * rng.lognormal(sigma = noise, size = 3)
name = "As{}_nu={}_K={}_gmax={}_q={}_m={}".format(len(nus), "_".join([str(x) for x in nus]), "_".join([str(x) for x in Ks]), gmax, q, m, )
print(name)

for c, conc in enumerate(concs):
    
    X0 = 1 if isinstance(nus, float) else len(nus)
    X0 = np.array([conc] * X0 + [0.1])
    
    sol = solve_ivp(Monod_addt,
                    (T[0], T[-1]),
                    X0,
                    t_eval = T,
                    args = (nus, Ks, gmax, q, m,),
                    )
    N = sol.y[-1]
    Nc = to_AUCs(T, N, t0=0)
    F = np.exp(-Nc)
    R = to_rhos(T, N)

    dataframe = pd.DataFrame({"T": T, "n": N, "Nc": Nc, "F": F, "U": np.exp(- T), "rho": R, "C": conc, }, )
    data = dataframe if c == 0 else pd.concat([data, dataframe])
    
    plt.plot(T, N)
plt.show()

data.to_csv("{}/data_simulated/{}.csv".format(cwd, name), sep=",", index=False)

## Additional simulations for symbolic regression

### Grid simulation of multiplicative Monod

In [ ]:
noise = 0.2

nus, Ks, gmax, q, m = 0.70248826, 0.19589854, 0.31826243, 0.00106463, 0.67793093 # fitted parameters
nus = np.array([nus, nus]) * rng.lognormal(sigma = noise, size = 2)
Ks = np.array([Ks, Ks]) * rng.lognormal(sigma = noise, size = 2)
gmax, q, m = np.array([gmax, q, m, ]) * rng.lognormal(sigma = noise, size = 3)
name = "Ms{}_nu={}_K={}_gmax={}_q={}_m={}".format(len(nus), "_".join([str(x) for x in nus]), "_".join([str(x) for x in Ks]), gmax, q, m, )
print(name)

for c1, conc1 in enumerate(concs):
    
    for c2, conc2 in enumerate(concs):
        
        X0 = np.array([conc1, conc2] + [0.1])
        
        sol = solve_ivp(Monod_mult,
                        (T[0], T[-1]),
                        X0,
                        t_eval = T,
                        args = (nus, Ks, gmax, q, m,),
                        )
        N = sol.y[-1]
        Nc = to_AUCs(T, N, t0=0)
        F = np.exp(-Nc)
        R = to_rhos(T, N)
    
        dataframe = pd.DataFrame({"T": T, "n": N, "Nc": Nc, "F": F, "U": np.exp(- T), "rho": R, "C1": conc1, "C2": conc2, }, )
        data = dataframe if (c1 == 0 and c2 == 0) else pd.concat([data, dataframe])
    
    plt.plot(T, N)
plt.show()

data.to_csv("{}/data_simulated_grid/{}.csv".format(cwd, name), sep=",", index=False)

### Grid simulation of additive Monod

In [ ]:
noise = 0.2

nus, Ks, gmax, q, m = 8.38874989e-01, 3.86999496e+00, 1.04200753e+00, 5.42190219e-04, 6.74326085e-01 # fitted parameters
nus = np.array([nus, nus]) * rng.lognormal(sigma = noise, size = 2)
Ks = np.array([Ks, Ks]) * rng.lognormal(sigma = noise, size = 2)
gmax, q, m = np.array([gmax, q, m, ]) * rng.lognormal(sigma = noise, size = 3)
name = "As{}_nu={}_K={}_gmax={}_q={}_m={}".format(len(nus), "_".join([str(x) for x in nus]), "_".join([str(x) for x in Ks]), gmax, q, m, )
print(name)

for c1, conc1 in enumerate(concs):
    
    for c2, conc2 in enumerate(concs):
        
        X0 = np.array([conc1, conc2] + [0.1])
        
        sol = solve_ivp(Monod_addt,
                        (T[0], T[-1]),
                        X0,
                        t_eval = T,
                        args = (nus, Ks, gmax, q, m,),
                        )
        N = sol.y[-1]
        Nc = to_AUCs(T, N, t0=0)
        F = np.exp(-Nc)
        R = to_rhos(T, N)
    
        dataframe = pd.DataFrame({"T": T, "n": N, "Nc": Nc, "F": F, "U": np.exp(- T), "rho": R, "C1": conc1, "C2": conc2, }, )
        data = dataframe if (c1 == 0 and c2 == 0) else pd.concat([data, dataframe])
    
    plt.plot(T, N)
plt.show()

data.to_csv("{}/data_simulated_grid/{}.csv".format(cwd, name), sep=",", index=False)

In [ ]:
# simulation time

dt = 0.1
tmax = 50000
subtime = 500 # timepoint subsampling: 1/500

ts = np.arange(0, tmax + dt, dt) # timepoints for simulation
subts = ts[::subtime,np.newaxis]
T = np.repeat(subts, nconcs, axis=-1) # # timepoints for sub-sampling (ntimes, nconcs)
N = np.zeros(T.shape) # sub-sampled population sizes (ntimes, nconcs)

ntimes = ts.shape[0]
nsbtms = T.shape[0]

In [ ]:
# simulation

for resc in rescs:
    
    nu = nuxi[:resc,:]
    gm = gmix[:,:resc]
    K_ = K_ix[:,:resc]
    
    for modl in [fA, fM]:
        for strn in strns[:1]:
                
            name = "{}s{}i{}.csv".format(modl.__name__[1], resc, strn)
            
            for c, conc in enumerate(concs):
                print(name, c, "   ", end="\r")
                
                # intial condition
                S0 = np.ones(resc) * 0.1 * conc #/ (resc)
                N0 = np.zeros(nstrns)
                N0[strn] = 0.1
                X0 = np.concatenate([S0, N0])
                
                # Euler's method
                for t, time in enumerate(ts):
                    dX = RCM(t, X0, resc, modl, nu, gm, K_,) * dt # Euler step
                    dX[resc:] *= alpha_RCM(time, qm_i[0], qm_i[1]) # adjustment function alpha
                    X0 += dX
                    # record only if the time is to be kept in subtimes Ts
                    if time in subts:
                        N[np.argmax(subts==time),c] = np.sum(X0[resc:]) # (ntimes, nconcs)
            
            Nc = to_AUCs(T, N, t0=0)
            F = np.exp(-Nc)
            R = to_rhos(T, N)
            
            df = pd.DataFrame()
            df["t"] =   T.flatten(order="F")
            df["N"] =   N.flatten(order="F")
            df["Nc"] = Nc.flatten(order="F")
            df["F"] =   F.flatten(order="F")
            df["rho"] = R.flatten(order="F")
            df["C"] = np.repeat(concs, nsbtms)
            
            df.to_csv("{}/data/{}".format(cwd, name), sep=",", index=False)

# Add lfit-inferred adjustment function alpha as a new column

In [ ]:
from os import listdir

files = sorted(listdir("{}/data".format(cwd)))
lfit_results = pd.read_csv("{}/lfit/lfit_results.csv".format(cwd), sep=",", header=[0,1,2,3], index_col=[0], )

In [ ]:
for file in files:
    
    name = file[:file.index(".csv")]
    code = name[:name.index("s")]
    resc = name[name.index("s")+1:name.index("i")]
    strn = name[name.index("i")+1:]
    
    data = pd.read_csv("{}/data/{}".format(cwd, file), sep=",")
    data["inferred_alpha"] = data["C"].apply(lambda x: lfit_results.loc["q",(code,resc,strn,str(x))] ) # parameter q
    data["m"] = data["C"].apply(lambda x: lfit_results.loc["m",(code,resc,strn,str(x))] )
    data["inferred_alpha"] = data["inferred_alpha"] / (np.exp(- data["m"] * data["t"]) + data["inferred_alpha"] )
    data.drop("m", axis=1, inplace=True)

    data.to_csv("{}/data/{}".format(cwd, file), sep=",", index=False)
    print(name, "   ", end="\r")

# Add lfit-inferred single adjustment function alpha
# (one across all concentrations) as a new column

In [ ]:
from os import listdir

files = sorted(listdir("{}/data".format(cwd)))
lfit_results = pd.read_csv("{}/lfit/lfit_results_single_alpha.csv".format(cwd), index_col=[0], header=[0,1,2])

In [ ]:
for file in files:
    
    name = file[:file.index(".csv")]
    code = name[:name.index("s")]
    resc = name[name.index("s")+1:name.index("i")]
    strn = name[name.index("i")+1:]
    
    data = pd.read_csv("{}/data/{}".format(cwd, file), sep=",")
    q = lfit_results.loc["q",(code,resc,strn)] # parameter q
    m = lfit_results.loc["m",(code,resc,strn)] # parameter q
    data["inferred_single_alpha"] = q / (np.exp(- m * data["t"]) + q )

    data.to_csv("{}/data/{}".format(cwd, file), sep=",", index=False)
    print(name, "   ", end="\r")

# Simulation and saving of no-adjustment data

In [ ]:
# simulation time

dt = 0.1
tmax = 50000
subtime = 500 # timepoint subsampling: 1/500

ts = np.arange(0, tmax + dt, dt) # timepoints for simulation
subts = ts[::subtime,np.newaxis]
T = np.repeat(subts, nconcs, axis=-1) # # timepoints for sub-sampling (ntimes, nconcs)
N = np.zeros(T.shape) # sub-sampled population sizes (ntimes, nconcs)

ntimes = ts.shape[0]
nsbtms = T.shape[0]

In [ ]:
# simulation

for resc in rescs[-1:]:
    
    nu = nuxi[:resc,:]
    gm = gmix[:,:resc]
    K_ = K_ix[:,:resc]
    
    for modl in [fA, fM][-1:]:
        for strn in strns:
                
            name = "{}s{}i{}.csv".format(modl.__name__[1], resc, strn)
            
            for c, conc in enumerate(concs):
                print(name, c, "   ", end="\r")
                
                # intial condition
                S0 = np.ones(resc) * 0.1 * conc #/ (resc)
                N0 = np.zeros(nstrns)
                N0[strn] = 0.1
                X0 = np.concatenate([S0, N0])
                
                # Euler's method
                for t, time in enumerate(ts):
                    dX = RCM(t, X0, resc, modl, nu, gm, K_,) * dt # Euler step
#                    dX[resc:] *= alpha_RCM(time, qm_i[0], qm_i[1]) # adjustment function alpha
                    X0 += dX
                    # record only if the time is to be kept in subtimes Ts
                    if time in subts:
                        N[np.argmax(subts==time),c] = np.sum(X0[resc:]) # (ntimes, nconcs)
            
            Nc = to_AUCs(T, N, t0=0)
            F = np.exp(-Nc)
            R = to_rhos(T, N)
            
            df = pd.DataFrame()
            df["t"] =   T.flatten(order="F")
            df["N"] =   N.flatten(order="F")
            df["Nc"] = Nc.flatten(order="F")
            df["F"] =   F.flatten(order="F")
            df["rho"] = R.flatten(order="F")
            df["C"] = np.repeat(concs, nsbtms)
            
            df.to_csv("{}/cplm/data/{}".format(cwd, name), sep=",", index=False)

## Simulation and saving of gLV data with and without adjustment

In [ ]:
# resource-consumer model for simulation

def gLV(t, X, substrate_nb, gmix, Kix):
    
    S, N = X[:substrate_nb], X[substrate_nb:]
    dX = np.zeros(X.shape)
    dX[substrate_nb:] = gmix[:,0] * (S[0] * K_ix[:,0] - N) * N # dynamics of the population

    return dX

In [ ]:
# simulation time

dt = 0.1
tmax = 50000
subtime = 500 # timepoint subsampling: 1/500

ts = np.arange(0, tmax + dt, dt) # timepoints for simulation
subts = ts[::subtime,np.newaxis]
T = np.repeat(subts, nconcs, axis=-1) # # timepoints for sub-sampling (ntimes, nconcs)
N = np.zeros(T.shape) # sub-sampled population sizes (ntimes, nconcs)

ntimes = ts.shape[0]
nsbtms = T.shape[0]

In [ ]:
# simulation

for resc in rescs[:1]:
    
    nu = nuxi[:resc,:]
    gm = gmix[:,:resc]
    K_ = K_ix[:,:resc]
    
    for strn in strns:
                
        name = "{}s{}i{}.csv".format("G", 0, strn)
            
        for c, conc in enumerate(concs):
            print(name, c, "   ", end="\r")
                
            # intial condition
            S0 = np.ones(resc) * 0.1 * conc #/ (resc)
            N0 = np.zeros(nstrns)
            N0[strn] = 0.1
            X0 = np.concatenate([S0, N0])
                
            # Euler's method
            for t, time in enumerate(ts):
                dX = gLV(t, X0, resc, gm, K_,) * dt # Euler step
                dX[resc:] *= alpha_RCM(time, qm_i[0], qm_i[1]) # adjustment function alpha
                X0 += dX
                # record only if the time is to be kept in subtimes Ts
                if time in subts:
                    N[np.argmax(subts==time),c] = np.sum(X0[resc:]) # (ntimes, nconcs)
        
        Nc = to_AUCs(T, N, t0=0)
        F = np.exp(-Nc)
        R = to_rhos(T, N)
        
        df = pd.DataFrame()
        df["t"] =   T.flatten(order="F")
        df["N"] =   N.flatten(order="F")
        df["Nc"] = Nc.flatten(order="F")
        df["F"] =   F.flatten(order="F")
        df["rho"] = R.flatten(order="F")
        df["C"] = np.repeat(concs, nsbtms)
        
        df.to_csv("{}/cplm/data/{}".format(cwd, name), sep=",", index=False)

In [ ]:
# simulation

for resc in rescs[:1]:
    
    nu = nuxi[:resc,:]
    gm = gmix[:,:resc]
    K_ = K_ix[:,:resc]
    
    for strn in strns:
                
        name = "{}s{}i{}.csv".format("V", 0, strn)
            
        for c, conc in enumerate(concs):
            print(name, c, "   ", end="\r")
                
            # intial condition
            S0 = np.ones(resc) * 0.1 * conc #/ (resc)
            N0 = np.zeros(nstrns)
            N0[strn] = 0.1
            X0 = np.concatenate([S0, N0])
                
            # Euler's method
            for t, time in enumerate(ts):
                dX = gLV(t, X0, resc, gm, K_,) * dt # Euler step
#                dX[resc:] *= alpha_RCM(time, qm_i[0], qm_i[1]) # adjustment function alpha
                X0 += dX
                # record only if the time is to be kept in subtimes Ts
                if time in subts:
                    N[np.argmax(subts==time),c] = np.sum(X0[resc:]) # (ntimes, nconcs)
        
        Nc = to_AUCs(T, N, t0=0)
        F = np.exp(-Nc)
        R = to_rhos(T, N)
        
        df = pd.DataFrame()
        df["t"] =   T.flatten(order="F")
        df["N"] =   N.flatten(order="F")
        df["Nc"] = Nc.flatten(order="F")
        df["F"] =   F.flatten(order="F")
        df["rho"] = R.flatten(order="F")
        df["C"] = np.repeat(concs, nsbtms)
        
        df.to_csv("{}/cplm/data/{}".format(cwd, name), sep=",", index=False)